In [ ]:
"""Extract CVE publication dates and descriptions from JSON files (1999–2025) and save to CSV."""

import os
import json
import csv
import glob
from datetime import datetime

def parse_date(date_string):
    """Parse date from datePublic/datePublished fields, making timezone-naive."""
    try:
        if not date_string:
            return None
        if 'T' in date_string:
            dt = datetime.fromisoformat(date_string.replace('Z', '+00:00'))
            return dt.replace(tzinfo=None)
        else:
            return datetime.strptime(date_string, '%Y-%m-%d')
    except Exception:
        return None

def extract_description(data):
    """Extract full English description text from JSON CVE file."""
    try:
        descriptions = (
            data.get('containers', {})
                .get('cna', {})
                .get('descriptions', [])
        )
        # Concatenate all English 'value' fields
        desc_texts = [
            d.get('value', '').strip()
            for d in descriptions
            if d.get('lang', '').lower() == 'en'
        ]
        return " ".join(desc_texts).replace('\n', ' ').replace('\r', ' ').strip()
    except Exception:
        return ""

def process_json_files(data_dir, output_csv="cve_descriptions_1999_2025.csv"):
    """Extract date and description from all CVE JSON files (1999–2025)."""
    start_date = datetime(1999, 1, 1)
    end_date = datetime(2025, 12, 31)

    file_pattern = os.path.join(data_dir, "**", "*.json")
    records = []
    total_files = 0
    matched_records = 0

    for file_path in glob.glob(file_pattern, recursive=True):
        total_files += 1
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            # Extract publication date (try multiple fields)
            date_public = None
            try:
                date_public = data.get('containers', {}).get('cna', {}).get('datePublic')
                if not date_public:
                    date_public = data.get('cveMetadata', {}).get('datePublished')
            except (AttributeError, KeyError):
                pass

            date_obj = parse_date(date_public)
            if not date_obj:
                continue

            # Only keep within 1999–2025
            if not (start_date <= date_obj <= end_date):
                continue

            description = extract_description(data)
            if not description:
                continue

            records.append({
                "date_published": date_obj.strftime("%Y-%m-%d"),
                "description": description
            })
            matched_records += 1

            if matched_records % 1000 == 0:
                print(f" {matched_records} records extracted from {total_files} files...")

        except (json.JSONDecodeError, UnicodeDecodeError) as e:
            print(f"Skipped {file_path}: {e}")
            continue

    # Sort by date
    records.sort(key=lambda x: x['date_published'])

    # Save to CSV
    with open(output_csv, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=["date_published", "description"])
        writer.writeheader()
        writer.writerows(records)

    print(f"\nDone! Processed {total_files} files.")
    print(f"{matched_records} valid CVEs saved to {output_csv}")

def main():
    data_dir = "C:/Users/Bella/Documents/GitHub/cvelistV5/cves"  
    process_json_files(data_dir)

if __name__ == "__main__":
    main()


In [ ]:

"""Script to encode threat intelligence data (MITRE ATT&CK matrices and CVE descriptions) into vector embeddings using SentenceTransformers, and merge them into a unified embedding space."""
import os
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm
import numpy as np
import hashlib
import glob

# CONFIGURATION
INPUT_DIR = "Dissertation/Agentic LLM/RAG data"        # directory with raw files
INTERMEDIATE_DIR = "Dissertation/Agentic LLM/RAG data/embeddings_encoded_processed"  # per-file embeddings
OUTPUT_FILE = "data/unified_embeddings.csv" # final merged embedding space
MODEL_NAME = "all-MiniLM-L6-v2"           # good balance of speed and quality

# SETUP 
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

print(f"Loading SentenceTransformer model: {MODEL_NAME}")
model = SentenceTransformer(MODEL_NAME)

def load_file(path):
    if path.lower().endswith(".csv"):
        return pd.read_csv(path)
    elif path.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    else:
        raise ValueError("Unsupported file type")

def flatten_attack_matrix(df):
    """Flatten MITRE ATT&CK / CAPEC tables (tactic → technique)."""
    df.columns = df.columns.str.strip()
    records = []
    for tactic in df.columns:
        for val in df[tactic].dropna().unique():
            if isinstance(val, str) and val.strip():
                records.append(f"{tactic}: {val.strip()}")
    return records

def flatten_cve_descriptions(df):
    """Flatten CVE/description datasets with date + text columns."""
    date_col, desc_col = None, None
    for c in df.columns:
        cl = c.lower()
        if "date" in cl:
            date_col = c
        elif any(x in cl for x in ["desc", "summary", "detail", "text"]):
            desc_col = c

    if desc_col is None:
        desc_col = df.columns[-1]  

    records = []
    for _, row in df.iterrows():
        date = str(row[date_col]) if date_col in df.columns else ""
        text = str(row[desc_col]) if desc_col in df.columns else ""
        if isinstance(text, str) and text.strip():
            records.append(f"{date}: {text.strip()}" if date else text.strip())
    return records

# Encode each file 
for file in tqdm(os.listdir(INPUT_DIR), desc="Encoding threat files"):
    if not file.lower().endswith((".csv", ".xlsx", ".xls")):
        continue

    fpath = os.path.join(INPUT_DIR, file)
    print(f"\n📄 Processing {fpath}")

    try:
        df = load_file(fpath)

        # Decide encoding mode
        if any("tactic" in c.lower() or "reconnaissance" in c.lower() for c in df.columns):
            records = flatten_attack_matrix(df)
            mode = "ATTACK matrix"
        else:
            records = flatten_cve_descriptions(df)
            mode = "CVE text"

        if not records:
            print(f"No valid text found in {file}")
            continue

        print(f" Encoding {len(records)} {mode} entries ...")
        embeddings = model.encode(records, normalize_embeddings=True)

        # Save per-file embeddings
        emb_df = pd.DataFrame(embeddings)
        emb_df.insert(0, "text", records)
        emb_df.insert(1, "source_file", file)
        out_path = os.path.join(INTERMEDIATE_DIR, file.replace(".xlsx", "_encoded.csv")
                                                       .replace(".xls", "_encoded.csv")
                                                       .replace(".csv", "_encoded.csv"))
        emb_df.to_csv(out_path, index=False)
        print(f"Saved to {out_path}")

    except Exception as e:
        print(f" Error processing {file}: {e}")

# Merge and deduplicate all embeddings
print("\n Merging all embeddings into a unified space...")

all_files = glob.glob(os.path.join(INTERMEDIATE_DIR, "*_encoded.csv"))
merged_df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

# Add unique hash to identify duplicates
merged_df["hash"] = merged_df["text"].apply(lambda x: hashlib.md5(str(x).encode()).hexdigest())
merged_df.drop_duplicates(subset="hash", inplace=True)
merged_df.drop(columns=["hash"], inplace=True)

merged_df.reset_index(drop=True, inplace=True)
merged_df.to_csv(OUTPUT_FILE, index=False)

print(f" Unified embedding space saved to {OUTPUT_FILE}")
print(f" Total unique embedded entries: {len(merged_df)}")

